In [255]:
import re
import os

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from scipy.stats import wilcoxon

# Reading files from the directory

In [256]:
for dirname, _, filenames in os.walk('./input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

./input\for_analysis\results_synthetic\loss_accuracy\acrgnn_elu_loss_accuracy.csv
./input\for_analysis\results_synthetic\loss_accuracy\acrgnn_elu_loss_accuracy_quantized.csv
./input\for_analysis\results_synthetic\loss_accuracy\acrgnn_gelu_loss_accuracy.csv
./input\for_analysis\results_synthetic\loss_accuracy\acrgnn_gelu_loss_accuracy_quantized.csv
./input\for_analysis\results_synthetic\loss_accuracy\acrgnn_leakyrelu_loss_accuracy.csv
./input\for_analysis\results_synthetic\loss_accuracy\acrgnn_leakyrelu_loss_accuracy_quantized.csv
./input\for_analysis\results_synthetic\loss_accuracy\acrgnn_normalized_loss_accuracy.csv
./input\for_analysis\results_synthetic\loss_accuracy\acrgnn_normalized_loss_accuracy_quantized.csv
./input\for_analysis\results_synthetic\loss_accuracy\acrgnn_relu6_loss_accuracy.csv
./input\for_analysis\results_synthetic\loss_accuracy\acrgnn_relu6_loss_accuracy_quantized.csv
./input\for_analysis\results_synthetic\loss_accuracy\acrgnn_relu_loss_accuracy.csv
./input\for_ana

## [Reading data] Creating a file_path variable for convenience 

In [257]:
file_path = "./input/for_analysis/results_synthetic/loss_accuracy/"

## [Reading data] Creating a file_name variable for convenience 

In [258]:
file_name="_loss_accuracy.csv"
file_name_quantized="_loss_accuracy_quantized.csv"

# Function for writing keys

In [259]:
def print_keys(key):
    latex_map = {
        "p1": r"$p1: \alpha_1(x) := \exists^{[8,10]}y\left(\alpha_0(y) \wedge \neg E(x,y)\right)$",
        "p2": r"$p2: \alpha_2(x) := \exists^{[10,30]}y\left(\alpha_1(y) \wedge \neg E(x,y)\right)$",
        "p3": r"$p3: \alpha_3(x) := \exists^{[10,30]}y\left(\alpha_2(y) \wedge \neg E(x,y)\right)$"
    }
    clean_latex = re.sub(r'[\x00-\x1F\x7F]', '', latex_map.get(key, ''))
    return clean_latex

# Analysis of ACRGNN with different activation functions

##  [Original] Read datasets before quantization for different activation functions

In [260]:
df_acrgnn_relu_nonqua=pd.read_csv(f'{file_path}acrgnn_relu{file_name}')

df_acrgnn_relu6_nonqua=pd.read_csv(f'{file_path}acrgnn_relu6{file_name}')

df_acrgnn_trrelu_nonqua=pd.read_csv(f'{file_path}acrgnn_trrelu{file_name}')

df_acrgnn_leakyrelu_nonqua=pd.read_csv(f'{file_path}acrgnn_leakyrelu{file_name}')

df_acrgnn_gelu_nonqua=pd.read_csv(f'{file_path}acrgnn_gelu{file_name}')

df_acrgnn_silu_nonqua=pd.read_csv(f'{file_path}acrgnn_silu{file_name}')

df_acrgnn_sigmoid_nonqua=pd.read_csv(f'{file_path}acrgnn_sigmoid{file_name}')

df_acrgnn_normalized_nonqua=pd.read_csv(f'{file_path}acrgnn_normalized{file_name}')

df_acrgnn_softplus_nonqua=pd.read_csv(f'{file_path}acrgnn_softplus{file_name}')

df_acrgnn_elu_nonqua=pd.read_csv(f'{file_path}acrgnn_elu{file_name}')

In [261]:
dfs_nonqua = [df_acrgnn_relu_nonqua,
       df_acrgnn_relu6_nonqua,
       df_acrgnn_trrelu_nonqua,
       df_acrgnn_leakyrelu_nonqua,
       df_acrgnn_gelu_nonqua,
       df_acrgnn_silu_nonqua,
       df_acrgnn_sigmoid_nonqua,
       df_acrgnn_normalized_nonqua,
       df_acrgnn_softplus_nonqua,
       df_acrgnn_elu_nonqua]

In [262]:
activation_map = {
    "relu": "ReLU",
    "relu6": "ReLU6",
    "trrelu": "trReLU",
    "leakyrelu": "LeakyReLU",
    "gelu": "GELU",
    "silu": "SiLU",
    "sigmoid": "Sigmoid",
    "normalized": "Normalized",
    "softplus": "Softplus",
    "elu": "ELU"
}
for df in dfs_nonqua:
    df["activation_function"] = df["activation_function"].replace(activation_map)

In [263]:
df_acrgnn_relu_nonqua.head()

,activation_function,key,learning_rate,layer,hidden_dimension,train_loss,test1_loss,test2_loss,train_micro,train_macro,test1_micro,test1_macro,test2_micro,test2_macro
0,ReLU,p1,0.010,1,2,0.691826,0.691835,0.693356,0.527589,0.528842,0.525732,0.526691,0.512267,0.513652
1,ReLU,p1,0.001,1,2,0.693204,0.694440,0.697281,0.527652,0.528903,0.525776,0.526732,0.512519,0.513885
2,ReLU,p1,0.005,1,2,0.691823,0.691832,0.693340,0.527589,0.528842,0.525732,0.526691,0.512267,0.513652
3,ReLU,p1,0.010,2,2,0.687234,0.687746,0.694358,0.529953,0.531171,0.526483,0.527759,0.519725,0.519664
4,ReLU,p1,0.001,2,2,0.713006,0.714625,0.705160,0.478870,0.478160,0.480237,0.479855,0.489102,0.487713


## [Original] Information about the datasets

In [264]:
def combine_with_activation(dataframes):
    combined = []
    for df in dataframes:
        temp = df.copy()
        combined.append(temp[temp.columns])
    return pd.concat(combined, ignore_index=True)

In [265]:
combined_acrgnn_activationFunction_nonqua = combine_with_activation(dfs_nonqua)
combined_acrgnn_activationFunction_nonqua.head(500)

,activation_function,key,learning_rate,layer,hidden_dimension,train_loss,test1_loss,test2_loss,train_micro,train_macro,test1_micro,test1_macro,test2_micro,test2_macro
0,ReLU,p1,0.010,1,2,0.691826,0.691835,0.693356,0.527589,0.528842,0.525732,0.526691,0.512267,0.513652
1,ReLU,p1,0.001,1,2,0.693204,0.694440,0.697281,0.527652,0.528903,0.525776,0.526732,0.512519,0.513885
2,ReLU,p1,0.005,1,2,0.691823,0.691832,0.693340,0.527589,0.528842,0.525732,0.526691,0.512267,0.513652
3,ReLU,p1,0.010,2,2,0.687234,0.687746,0.694358,0.529953,0.531171,0.526483,0.527759,0.519725,0.519664
4,ReLU,p1,0.001,2,2,0.713006,0.714625,0.705160,0.478870,0.478160,0.480237,0.479855,0.489102,0.487713
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,ReLU,p2,0.010,6,9,0.407840,0.683972,0.830830,0.831227,0.833696,0.833772,0.836194,0.667868,0.669064
496,ReLU,p2,0.001,6,9,0.564143,0.548278,0.853796,0.757715,0.763609,0.778154,0.782913,0.479375,0.481275
497,ReLU,p2,0.005,6,9,0.423789,0.403765,0.686647,0.826276,0.828917,0.829939,0.832503,0.726123,0.724689
498,ReLU,p2,0.010,7,9,0.404422,0.385495,0.510845,0.831981,0.834372,0.837025,0.839275,0.777894,0.777366


In [266]:
combined_acrgnn_activationFunction_nonqua.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9000 entries, 0 to 8999
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   activation_function  9000 non-null   object 
 1   key                  9000 non-null   object 
 2   learning_rate        9000 non-null   float64
 3   layer                9000 non-null   int64  
 4   hidden_dimension     9000 non-null   int64  
 5   train_loss           9000 non-null   float64
 6   test1_loss           9000 non-null   float64
 7   test2_loss           9000 non-null   float64
 8   train_micro          9000 non-null   float64
 9   train_macro          9000 non-null   float64
 10  test1_micro          9000 non-null   float64
 11  test1_macro          9000 non-null   float64
 12  test2_micro          9000 non-null   float64
 13  test2_macro          9000 non-null   float64
dtypes: float64(10), int64(2), object(2)
memory usage: 984.5+ KB


In [267]:
combined_acrgnn_activationFunction_nonqua.hidden_dimension.unique(), combined_acrgnn_activationFunction_nonqua.learning_rate.unique()        

(array([ 2,  4,  5,  6,  7,  8,  9, 10, 16, 32]), array([0.01 , 0.001, 0.005]))

In [268]:
combined_acrgnn_activationFunction_nonqua['activation_function'].unique()

array(['ReLU', 'ReLU6', 'trReLU', 'LeakyReLU', 'GELU', 'SiLU', 'Sigmoid',
       'Normalized', 'Softplus', 'ELU'], dtype=object)

## [Original] Find the best value of the learning rate based on the accuracy

In [269]:
df=combined_acrgnn_activationFunction_nonqua.copy()
df_grouped = (
    df.groupby([
        "activation_function",
        "key",
        "layer",
        "hidden_dimension",
        "learning_rate"
    ])
    .agg(
        mean_test1_micro=("test1_micro", "mean"),
        std_test1_micro=("test1_micro", "std"),
        runs=("test1_micro", "count")
    )
    .reset_index()
)
df_grouped

,activation_function,key,layer,hidden_dimension,learning_rate,mean_test1_micro,std_test1_micro,runs
0,ELU,p1,1,2,0.001,0.530905,NaN,1
1,ELU,p1,1,2,0.005,0.506897,NaN,1
2,ELU,p1,1,2,0.010,0.774295,NaN,1
3,ELU,p1,1,4,0.001,0.515165,NaN,1
4,ELU,p1,1,4,0.005,0.760898,NaN,1
...,...,...,...,...,...,...,...,...
8995,trReLU,p3,10,16,0.005,0.558485,NaN,1
8996,trReLU,p3,10,16,0.010,0.561593,NaN,1
8997,trReLU,p3,10,32,0.001,0.521374,NaN,1
8998,trReLU,p3,10,32,0.005,0.565144,NaN,1


In [270]:
best_lr = df_grouped.loc[
    df_grouped.groupby([
        "activation_function",
        "key",
        "layer",
        "hidden_dimension"
    ])["mean_test1_micro"].idxmax()
].reset_index(drop=True)
best_lr

,activation_function,key,layer,hidden_dimension,learning_rate,mean_test1_micro,std_test1_micro,runs
0,ELU,p1,1,2,0.010,0.774295,NaN,1
1,ELU,p1,1,4,0.010,0.807631,NaN,1
2,ELU,p1,1,5,0.010,0.902379,NaN,1
3,ELU,p1,1,6,0.010,0.825537,NaN,1
4,ELU,p1,1,7,0.010,0.786984,NaN,1
...,...,...,...,...,...,...,...,...
2995,trReLU,p3,10,8,0.005,0.566698,NaN,1
2996,trReLU,p3,10,9,0.005,0.572380,NaN,1
2997,trReLU,p3,10,10,0.010,0.560172,NaN,1
2998,trReLU,p3,10,16,0.001,0.574555,NaN,1


In [271]:
global_lr_stats = (
    df.groupby("learning_rate")
      .agg(
          mean_test1_micro=("test1_micro", "mean"),
          std_test1_micro=("test1_micro", "std"),
          runs=("test1_micro", "count")
      )
      .reset_index()
      .sort_values("mean_test1_micro", ascending=False)
)
global_lr_stats

,learning_rate,mean_test1_micro,std_test1_micro,runs
2,0.010,0.692475,0.099803,3000
1,0.005,0.671970,0.085466,3000
0,0.001,0.608216,0.068278,3000


In [272]:
lambda_val = 0.5

global_lr_stats["robust_score"] = (
    global_lr_stats["mean_test1_micro"]
    - lambda_val * global_lr_stats["std_test1_micro"]
)

global_lr_stats = global_lr_stats.sort_values(
    "robust_score", ascending=False
)

global_lr_stats

,learning_rate,mean_test1_micro,std_test1_micro,runs,robust_score
2,0.010,0.692475,0.099803,3000,0.642573
1,0.005,0.671970,0.085466,3000,0.629237
0,0.001,0.608216,0.068278,3000,0.574078


In [273]:
best_global_lr = global_lr_stats.iloc[0]["learning_rate"]
print("Best global LR:", best_global_lr)

Best global LR: 0.01


In [274]:
df_best_lr = df[df["learning_rate"] == best_global_lr].copy()

print("Original rows:", len(df))
print("Filtered rows:", len(df_best_lr))

Original rows: 9000
Filtered rows: 3000


In [275]:
df_best_lr = df[np.isclose(df["learning_rate"], best_global_lr)]
df_best_lr.head()

,activation_function,key,learning_rate,layer,hidden_dimension,train_loss,test1_loss,test2_loss,train_micro,train_macro,test1_micro,test1_macro,test2_micro,test2_macro
0,ReLU,p1,0.01,1,2,0.691826,0.691835,0.693356,0.527589,0.528842,0.525732,0.526691,0.512267,0.513652
3,ReLU,p1,0.01,2,2,0.687234,0.687746,0.694358,0.529953,0.531171,0.526483,0.527759,0.519725,0.519664
6,ReLU,p1,0.01,3,2,0.686329,0.687378,0.711340,0.538190,0.538804,0.532806,0.533406,0.511078,0.512456
9,ReLU,p1,0.01,4,2,0.687210,0.687344,0.695917,0.529864,0.531089,0.531037,0.532077,0.525309,0.525865
12,ReLU,p1,0.01,5,2,0.690692,0.690920,0.692208,0.526132,0.527480,0.522637,0.523876,0.509853,0.510703


In [276]:
df_best_lr.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3000 entries, 0 to 8997
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   activation_function  3000 non-null   object 
 1   key                  3000 non-null   object 
 2   learning_rate        3000 non-null   float64
 3   layer                3000 non-null   int64  
 4   hidden_dimension     3000 non-null   int64  
 5   train_loss           3000 non-null   float64
 6   test1_loss           3000 non-null   float64
 7   test2_loss           3000 non-null   float64
 8   train_micro          3000 non-null   float64
 9   train_macro          3000 non-null   float64
 10  test1_micro          3000 non-null   float64
 11  test1_macro          3000 non-null   float64
 12  test2_micro          3000 non-null   float64
 13  test2_macro          3000 non-null   float64
dtypes: float64(10), int64(2), object(2)
memory usage: 351.6+ KB


## [Original] Find the best value of the hidden dimension based on the best learning rate

In [277]:
hidden_stats = (
    df_best_lr.groupby("hidden_dimension")
      .agg(
          mean_test1_micro=("test1_micro", "mean"),
          std_test1_micro=("test1_micro", "std"),
          runs=("test1_micro", "count")
      )
      .reset_index()
      .sort_values("mean_test1_micro", ascending=False)
)

hidden_stats

,hidden_dimension,mean_test1_micro,std_test1_micro,runs
9,32,0.736638,0.111421,300
8,16,0.727891,0.109303,300
6,9,0.715254,0.103112,300
7,10,0.714753,0.105592,300
4,7,0.698009,0.100052,300
5,8,0.692741,0.088101,300
3,6,0.686096,0.086324,300
2,5,0.675784,0.087929,300
1,4,0.664589,0.070882,300
0,2,0.612992,0.063866,300


In [278]:
lambda_val = 0.5

hidden_stats["robust_score"] = (
    hidden_stats["mean_test1_micro"]
    - lambda_val * hidden_stats["std_test1_micro"]
)

hidden_stats = hidden_stats.sort_values("robust_score", ascending=False)

hidden_stats

,hidden_dimension,mean_test1_micro,std_test1_micro,runs,robust_score
9,32,0.736638,0.111421,300,0.680928
8,16,0.727891,0.109303,300,0.673239
6,9,0.715254,0.103112,300,0.663698
7,10,0.714753,0.105592,300,0.661957
5,8,0.692741,0.088101,300,0.648690
4,7,0.698009,0.100052,300,0.647983
3,6,0.686096,0.086324,300,0.642934
2,5,0.675784,0.087929,300,0.631820
1,4,0.664589,0.070882,300,0.629148
0,2,0.612992,0.063866,300,0.581059


In [279]:
best_hidden = hidden_stats.iloc[0]["hidden_dimension"]
print("Best global hidden dimension:", best_hidden)

Best global hidden dimension: 32.0


## [Original]  Accuracy across 10 layers for 10 Activation functions. Heatmap plot

In [280]:
def heatmap_grid(df: pd.DataFrame,
                 metrics=("train_micro", "test1_micro", "test2_micro"),
                 keys=("p1", "p2", "p3"),
                 activation_order=('ReLU', 'ReLU6', 'trReLU', 'LeakyReLU', 'GELU', 'SiLU', 'Sigmoid',
       'Normalized', 'Softplus', 'ELU'),
                 percent=True):
    # Make subplot grid: rows = metrics, cols = keys
    fig = make_subplots(
        rows=len(metrics), cols=len(keys),
        subplot_titles=[f"{m.split('_')[0].capitalize()} – {k}" for m in metrics for k in keys],
        horizontal_spacing=0.05, vertical_spacing=0.08
    )

    # Ensure consistent order for activations
    df = df.copy()
    df["activation_function"] = pd.Categorical(df["activation_function"], activation_order, ordered=True)

    # Iterate over metrics and keys
    for i, m in enumerate(metrics, start=1):
        for j, k in enumerate(keys, start=1):
            d = df[df["key"] == k]
            z = d.pivot(index="layer", columns="activation_function", values=m).sort_index()
            if percent:
                z = z * 100

            heatmap = go.Heatmap(
                z=z.values,
                x=z.columns.astype(str),
                y=z.index.astype(str),
                coloraxis="coloraxis",
                hovertemplate=f"Layer: %{{y}}<br>Activation: %{{x}}<br>{m}: %{{z:.2f}}<extra></extra>"
            )
            fig.add_trace(heatmap, row=i, col=j)

    # Shared color scale
    fig.update_layout(
        title="ACR-GNN Accuracy Heatmaps (Layers × Activations)",
        coloraxis=dict(colorscale="Viridis", colorbar=dict(title="Accuracy (%)")),
        height=1200, width=1200
    )

    return fig

In [281]:
heatmap_grid(df_best_lr[df_best_lr.hidden_dimension == best_hidden]).show()

We deecided to procede after with the layer from 1 to 3

In [282]:
cols = ["train_micro", "test1_micro", "test2_micro"]
df_selected=df_best_lr[df_best_lr.hidden_dimension == best_hidden].copy()
for i in range(1,4):
    df_percent = df_selected[(df_selected['layer']==i) ].drop(columns=['layer','train_loss','test1_loss','test2_loss','train_macro','test1_macro','test2_macro'])
    df_percent[cols] = df_percent[cols].mul(100).round(1)  # % format
    
    activation_order = ['ReLU', 'ReLU6', 'trReLU', 'LeakyReLU', 'GELU', 'SiLU', 'Sigmoid',
       'Normalized', 'Softplus', 'ELU']
    
    # Make 'activation_function' a categorical column with this order
    df_percent["activation_function"] = pd.Categorical(
        df_percent["activation_function"],
        categories=activation_order,
        ordered=True
    )
    
    # Sort according to the defined order
    df_percent = df_percent.sort_values("activation_function")
    
    
    
    # Pivot so that activation_function is index and keys become columns
    table_df = df_percent.pivot_table(
        index="activation_function",
        columns="key",
        values=cols,
        observed=False
    )
    
    # Reorder columns so p1, p2, p3 are grouped together
    table_df = table_df[[("train_micro", "p1"), ("test1_micro", "p1"), ("test2_micro", "p1"),
                         ("train_micro", "p2"), ("test1_micro", "p2"), ("test2_micro", "p2"),
                         ("train_micro", "p3"), ("test1_micro", "p3"), ("test2_micro", "p3")]]
    
    # Convert to LaTeX
    latex_str = table_df.to_latex(
        index=True,
        header=True,
        multicolumn=True,
        multirow=False,
        escape=False,
        column_format="c ccc ccc ccc",
        float_format="%.1f\\%%"  # formats numbers like 96.9%
    )
    print(i)
    print(latex_str)


1
\begin{tabular}{c ccc ccc ccc}
\toprule
 & train_micro & test1_micro & test2_micro & train_micro & test1_micro & test2_micro & train_micro & test1_micro & test2_micro \\
key & p1 & p1 & p1 & p2 & p2 & p2 & p3 & p3 & p3 \\
activation_function &  &  &  &  &  &  &  &  &  \\
\midrule
ReLU & 95.9\% & 95.4\% & 73.3\% & 69.2\% & 70.3\% & 63.2\% & 69.3\% & 68.6\% & 74.4\% \\
ReLU6 & 98.9\% & 98.4\% & 74.5\% & 69.5\% & 70.5\% & 62.2\% & 69.0\% & 68.5\% & 72.4\% \\
trReLU & 96.0\% & 95.5\% & 78.6\% & 76.5\% & 77.7\% & 50.9\% & 72.5\% & 72.0\% & 69.2\% \\
LeakyReLU & 94.9\% & 94.2\% & 69.6\% & 69.3\% & 70.5\% & 62.4\% & 69.3\% & 68.8\% & 74.4\% \\
GELU & 99.1\% & 98.6\% & 83.7\% & 75.9\% & 77.2\% & 50.1\% & 73.1\% & 72.3\% & 76.1\% \\
SiLU & 99.9\% & 99.9\% & 92.6\% & 73.5\% & 75.1\% & 48.9\% & 71.2\% & 70.4\% & 72.6\% \\
Sigmoid & 98.0\% & 97.4\% & 82.1\% & 71.6\% & 72.8\% & 53.8\% & 70.6\% & 69.9\% & 62.0\% \\
Normalized & 96.5\% & 96.0\% & 73.5\% & 71.0\% & 72.1\% & 51.7\% & 70.6\% & 70.2\% 

## [Original] Generalization Ratio and Gap

In [283]:
df_generalization_ratio = df_selected.drop(columns=['train_loss','test1_loss','test2_loss','train_macro','test1_macro','test2_macro'])

df_generalization_ratio['GR_Test1']=df_generalization_ratio['test1_micro'].mul(100)/df_generalization_ratio['train_micro'].mul(100)
# df_generalization_ratio['GR_Test1']=df_generalization_ratio['GR_Test1'].round(2)
# df_generalization_ratio['GR_error_Test1']=(1-df_generalization_ratio['test1_micro'])/(1-df_generalization_ratio['train_micro'])
df_generalization_ratio['GGap_Test1']=df_generalization_ratio['train_micro'].mul(100)-df_generalization_ratio['test1_micro'].mul(100)

df_generalization_ratio['GR_Test2']=df_generalization_ratio['test2_micro'].mul(100)/df_generalization_ratio['train_micro'].mul(100)
# df_generalization_ratio['GR_Test2']=df_generalization_ratio['GR_Test2'].round(2)
# df_generalization_ratio['GR_error_Test2']=(1-df_generalization_ratio['test2_micro'])/(1-df_generalization_ratio['train_micro'])
df_generalization_ratio['GGap_Test2']=df_generalization_ratio['train_micro'].mul(100)-df_generalization_ratio['test2_micro'].mul(100)


df_generalization_ratio_all_l=df_generalization_ratio.drop(columns=['train_micro','test1_micro','test2_micro'])
df_generalization_ratio[df_generalization_ratio['layer']==2]

,activation_function,key,learning_rate,layer,hidden_dimension,train_micro,test1_micro,test2_micro,GR_Test1,GGap_Test1,GR_Test2,GGap_Test2
273,ReLU,p1,0.01,2,32,1.000000,1.000000,0.991786,1.000000,0.000000,0.991786,0.821414
573,ReLU,p2,0.01,2,32,0.830942,0.838852,0.700292,1.009519,-0.790955,0.842768,13.065063
873,ReLU,p3,0.01,2,32,0.762900,0.759977,0.764345,0.996168,0.292307,1.001894,-0.144501
1173,ReLU6,p1,0.01,2,32,0.999982,1.000000,0.980437,1.000018,-0.001777,0.980455,1.954486
1473,ReLU6,p2,0.01,2,32,0.806927,0.813583,0.723205,1.008249,-0.665638,0.896246,8.372215
1773,ReLU6,p3,0.01,2,32,0.758131,0.753718,0.769614,0.994179,0.441327,1.015146,-1.148283
2073,trReLU,p1,0.01,2,32,0.998498,0.998143,0.955651,0.999644,0.035528,0.957088,4.284752
2373,trReLU,p2,0.01,2,32,0.787134,0.806408,0.556796,1.024487,-1.927439,0.707372,23.033762
2673,trReLU,p3,0.01,2,32,0.747432,0.749279,0.728257,1.002471,-0.184663,0.974345,1.917506
2973,LeakyReLU,p1,0.01,2,32,1.000000,1.000000,0.989192,1.000000,0.000000,0.989192,1.080808


## [Original]. LaTex tables "Generalization performance of the #-layer ACR-GNN with different activation functions (A/F), reported as both Generalization Ratios (Test/Train) and Generalization Gaps (Train – Test accuracy)."

In [284]:
activation_order = ['ReLU', 'ReLU6', 'trReLU', 'LeakyReLU', 'GELU', 'SiLU', 'Sigmoid','Normalized', 'Softplus', 'ELU']
cols = ["GR_Test1", "GGap_Test1", "GR_Test2","GGap_Test2"]
for i in range(1,11):
    df_generalization_ratio=df_generalization_ratio_all_l[(df_generalization_ratio_all_l['layer']==i) ].copy()
    # Make 'activation_function' a categorical column with this order
    df_generalization_ratio["activation_function"] = pd.Categorical(
        df_generalization_ratio["activation_function"],
        categories=activation_order,
        ordered=True
    )
    
    # Sort according to the defined order
    df_generalization_ratio = df_generalization_ratio.sort_values("activation_function")
    
    
    
    # Pivot so that activation_function is index and keys become columns
    table_df = df_generalization_ratio.pivot_table(
        index="activation_function",
        columns="key",
        values=cols,
        observed=False
    )
    #print(table_df)
    # Reorder columns so p1, p2, p3 are grouped together
    table_df = table_df[[("GR_Test1", "p1"), ("GGap_Test1", "p1"),("GR_Test2", "p1"), ("GGap_Test2", "p1"),
                         ("GR_Test1", "p2"), ("GGap_Test1", "p2"),("GR_Test2", "p2"), ("GGap_Test2", "p2"),
                         ("GR_Test1", "p3"), ("GGap_Test1", "p3"),("GR_Test2", "p3"), ("GGap_Test2", "p3")]].round(4)
    
    # Convert to LaTeX
    latex_str = table_df.to_latex(
        index=True,
        header=True,
        multicolumn=True,
        multirow=False,
        escape=False,
        column_format="c ccc ccc ccc ccc",
        caption=f"Generalization performance of the {i}-layer ACR-GNN with different activation functions (A/F), reported as both Generalization Ratios (Test/Train) and Generalization Gaps (Train – Test accuracy) across datasets $p_1$, $p_2$, and $p_3$.",
        label=f"tab:results:nonqua:ACR-GNN:{i}layer",
        float_format="%.3f"  # formats numbers like 96.9%
    )
    # print("----", i)
    # print(latex_str)

# Dynamic Post-Training Quantization

##  [dPTQ] Read datasets before quantization for different activation functions

In [285]:
df_acrgnn_relu_qua=pd.read_csv(f'{file_path}acrgnn_relu{file_name_quantized}')

df_acrgnn_relu6_qua=pd.read_csv(f'{file_path}acrgnn_relu6{file_name_quantized}')

df_acrgnn_trrelu_qua=pd.read_csv(f'{file_path}acrgnn_trrelu{file_name_quantized}')

df_acrgnn_leakyrelu_qua=pd.read_csv(f'{file_path}acrgnn_leakyrelu{file_name_quantized}')

df_acrgnn_gelu_qua=pd.read_csv(f'{file_path}acrgnn_gelu{file_name_quantized}')

df_acrgnn_silu_qua=pd.read_csv(f'{file_path}acrgnn_silu{file_name_quantized}')

df_acrgnn_sigmoid_qua=pd.read_csv(f'{file_path}acrgnn_sigmoid{file_name_quantized}')

df_acrgnn_normalized_qua=pd.read_csv(f'{file_path}acrgnn_normalized{file_name_quantized}')

df_acrgnn_softplus_qua=pd.read_csv(f'{file_path}acrgnn_softplus{file_name_quantized}')

df_acrgnn_elu_qua=pd.read_csv(f'{file_path}acrgnn_elu{file_name_quantized}')

In [286]:
dfs_qua = [df_acrgnn_relu_qua,
       df_acrgnn_relu6_qua,
       df_acrgnn_trrelu_qua,
       df_acrgnn_leakyrelu_qua,
       df_acrgnn_gelu_qua,
       df_acrgnn_silu_qua,
       df_acrgnn_sigmoid_qua,
       df_acrgnn_normalized_qua,
       df_acrgnn_softplus_qua,
       df_acrgnn_elu_qua]

In [287]:
for df in dfs_qua:
    df["activation_function"] = df["activation_function"].replace(activation_map)

In [288]:
df_acrgnn_relu_qua.head()

,activation_function,key,learning_rate,layer,hidden_dimension,bits,size_original,size_quantized,time_original,time_quantized,test1_loss,test2_loss,train_micro,train_macro,test1_micro,test1_macro,test2_micro,test2_macro
0,ReLU,p1,0.010,1,2,8,0.004796,0.00750,0.3,0.3,0.691835,0.693356,0.527589,0.528842,0.525732,0.526691,0.512267,0.513652
1,ReLU,p1,0.001,1,2,8,0.004796,0.00750,0.3,0.3,0.695348,0.697891,0.527652,0.528903,0.525776,0.526732,0.512411,0.513785
2,ReLU,p1,0.005,1,2,8,0.004796,0.00750,0.3,0.3,0.691832,0.693336,0.527589,0.528842,0.525732,0.526691,0.512267,0.513652
3,ReLU,p1,0.010,2,2,8,0.007994,0.01303,0.4,0.4,0.687772,0.694420,0.529389,0.530615,0.526085,0.527352,0.518572,0.518400
4,ReLU,p1,0.001,2,2,8,0.007994,0.01303,0.4,0.4,0.723570,0.705201,0.494744,0.494600,0.476346,0.475555,0.491912,0.490687


In [289]:
combined_acrgnn_activationFunction_qua = combine_with_activation(dfs_qua)
combined_acrgnn_activationFunction_qua.head(500)

,activation_function,key,learning_rate,layer,hidden_dimension,bits,size_original,size_quantized,time_original,time_quantized,test1_loss,test2_loss,train_micro,train_macro,test1_micro,test1_macro,test2_micro,test2_macro
0,ReLU,p1,0.010,1,2,8,0.004796,0.007500,0.3,0.3,0.691835,0.693356,0.527589,0.528842,0.525732,0.526691,0.512267,0.513652
1,ReLU,p1,0.001,1,2,8,0.004796,0.007500,0.3,0.3,0.695348,0.697891,0.527652,0.528903,0.525776,0.526732,0.512411,0.513785
2,ReLU,p1,0.005,1,2,8,0.004796,0.007500,0.3,0.3,0.691832,0.693336,0.527589,0.528842,0.525732,0.526691,0.512267,0.513652
3,ReLU,p1,0.010,2,2,8,0.007994,0.013030,0.4,0.4,0.687772,0.694420,0.529389,0.530615,0.526085,0.527352,0.518572,0.518400
4,ReLU,p1,0.001,2,2,8,0.007994,0.013030,0.4,0.4,0.723570,0.705201,0.494744,0.494600,0.476346,0.475555,0.491912,0.490687
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,ReLU,p2,0.010,6,8,8,0.025586,0.035604,1.5,1.4,0.524698,1.232839,0.734135,0.738947,0.728063,0.733027,0.507656,0.505560
496,ReLU,p2,0.001,6,8,8,0.025586,0.035604,1.5,1.5,0.663758,0.726221,0.608722,0.612452,0.628593,0.631860,0.587275,0.585586
497,ReLU,p2,0.005,6,8,8,0.025586,0.035604,1.5,1.5,0.559026,0.757880,0.694576,0.699683,0.699897,0.705745,0.626401,0.626059
498,ReLU,p2,0.010,7,8,8,0.029680,0.041215,1.7,1.7,0.559823,0.662867,0.706328,0.711274,0.698783,0.704045,0.593760,0.592464


In [290]:
combined_acrgnn_activationFunction_qua.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9000 entries, 0 to 8999
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   activation_function  9000 non-null   object 
 1   key                  9000 non-null   object 
 2   learning_rate        9000 non-null   float64
 3   layer                9000 non-null   int64  
 4   hidden_dimension     9000 non-null   int64  
 5   bits                 9000 non-null   int64  
 6   size_original        9000 non-null   float64
 7   size_quantized       9000 non-null   float64
 8   time_original        9000 non-null   float64
 9   time_quantized       9000 non-null   float64
 10  test1_loss           9000 non-null   float64
 11  test2_loss           9000 non-null   float64
 12  train_micro          9000 non-null   float64
 13  train_macro          9000 non-null   float64
 14  test1_micro          9000 non-null   float64
 15  test1_macro          9000 non-null   f

In [291]:
df_qua_heatmap = combined_acrgnn_activationFunction_qua[(combined_acrgnn_activationFunction_qua.hidden_dimension == best_hidden) & (combined_acrgnn_activationFunction_qua.learning_rate == best_global_lr)]
heatmap_grid(df_qua_heatmap).show()

In [292]:
df_nonqua_heatmap = df_best_lr[df_best_lr.hidden_dimension == best_hidden]
df_nonqua_heatmap

,activation_function,key,learning_rate,layer,hidden_dimension,train_loss,test1_loss,test2_loss,train_micro,train_macro,test1_micro,test1_macro,test2_micro,test2_macro
270,ReLU,p1,0.01,1,32,0.139133,0.143348,0.700049,0.958785,0.959055,0.954107,0.954761,0.733437,0.736938
273,ReLU,p1,0.01,2,32,0.035913,0.019126,0.052871,1.000000,1.000000,1.000000,1.000000,0.991786,0.992214
276,ReLU,p1,0.01,3,32,0.137448,0.133672,0.431175,0.962561,0.962783,0.960518,0.960758,0.874266,0.876611
279,ReLU,p1,0.01,4,32,0.577747,0.583102,0.643537,0.697604,0.698367,0.693563,0.694542,0.635948,0.638122
282,ReLU,p1,0.01,5,32,0.592249,0.595247,0.616150,0.685294,0.685912,0.682686,0.683639,0.666174,0.666721
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8985,ELU,p3,0.01,6,32,0.439118,0.437320,0.547992,0.784899,0.791961,0.775691,0.782508,0.757777,0.754226
8988,ELU,p3,0.01,7,32,0.454670,0.451417,0.669321,0.770650,0.778216,0.772229,0.779162,0.755106,0.750707
8991,ELU,p3,0.01,8,32,0.457543,0.456735,0.618923,0.768007,0.775907,0.756692,0.764317,0.768676,0.764555
8994,ELU,p3,0.01,9,32,0.472284,0.475162,1.691361,0.766214,0.773489,0.756914,0.764391,0.479682,0.484409


In [293]:
df_qua_heatmap

,activation_function,key,learning_rate,layer,hidden_dimension,bits,size_original,size_quantized,time_original,time_quantized,test1_loss,test2_loss,train_micro,train_macro,test1_micro,test1_macro,test2_micro,test2_macro
810,ReLU,p1,0.01,1,32,8,0.017532,0.010828,0.2,0.2,0.146457,0.751141,0.956314,0.957515,0.949421,0.950531,0.720323,0.724016
813,ReLU,p1,0.01,2,32,8,0.033274,0.019750,0.3,0.3,0.020086,0.050400,1.000000,1.000000,1.000000,1.000000,0.992362,0.992756
816,ReLU,p1,0.01,3,32,8,0.049400,0.028672,0.3,0.3,0.137727,0.454687,0.961486,0.961712,0.957600,0.957941,0.871456,0.873860
819,ReLU,p1,0.01,4,32,8,0.065462,0.037658,0.4,0.4,0.583949,0.649887,0.697396,0.698149,0.692236,0.693260,0.632237,0.634431
822,ReLU,p1,0.01,5,32,8,0.081524,0.046580,0.4,0.4,0.595776,0.616790,0.685502,0.686152,0.683350,0.684319,0.668336,0.668899
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8985,ELU,p3,0.01,6,32,8,0.097586,0.055572,0.5,0.5,0.440860,0.545012,0.781687,0.788568,0.772584,0.779306,0.762035,0.758557
8988,ELU,p3,0.01,7,32,8,0.113712,0.064575,0.5,0.6,0.454355,0.671866,0.768496,0.776078,0.769832,0.776657,0.752183,0.747743
8991,ELU,p3,0.01,8,32,8,0.129774,0.073514,0.6,0.6,0.459949,0.610032,0.766401,0.774373,0.755005,0.762882,0.769036,0.764964
8994,ELU,p3,0.01,9,32,8,0.145837,0.082517,0.6,0.7,0.475026,1.627524,0.767945,0.775117,0.757580,0.764944,0.507073,0.511512


In [294]:
cols = ["train_micro", "test1_micro", "test2_micro"]
df_qua_selected=df_qua_heatmap.copy()
for i in range(1,4):
    df_percent = df_qua_selected[(df_qua_selected['layer']==i) ].drop(columns=['layer','test1_loss','test2_loss','train_macro','test1_macro','test2_macro'])
    df_percent[cols] = df_percent[cols].mul(100).round(1)  # % format
    
    activation_order = ['ReLU', 'ReLU6', 'trReLU', 'LeakyReLU', 'GELU', 'SiLU', 'Sigmoid',
       'Normalized', 'Softplus', 'ELU']
    
    # Make 'activation_function' a categorical column with this order
    df_percent["activation_function"] = pd.Categorical(
        df_percent["activation_function"],
        categories=activation_order,
        ordered=True
    )
    
    # Sort according to the defined order
    df_percent = df_percent.sort_values("activation_function")
    
    
    
    # Pivot so that activation_function is index and keys become columns
    table_df = df_percent.pivot_table(
        index="activation_function",
        columns="key",
        values=cols,
        observed=False
    )
    
    # Reorder columns so p1, p2, p3 are grouped together
    table_df = table_df[[("train_micro", "p1"), ("test1_micro", "p1"), ("test2_micro", "p1"),
                         ("train_micro", "p2"), ("test1_micro", "p2"), ("test2_micro", "p2"),
                         ("train_micro", "p3"), ("test1_micro", "p3"), ("test2_micro", "p3")]]
    
    # Convert to LaTeX
    latex_str = table_df.to_latex(
        index=True,
        header=True,
        multicolumn=True,
        multirow=False,
        escape=False,
        column_format="c ccc ccc ccc",
        float_format="%.1f\\%%"  # formats numbers like 96.9%
    )
    print(i)
    print(latex_str)


1
\begin{tabular}{c ccc ccc ccc}
\toprule
 & train_micro & test1_micro & test2_micro & train_micro & test1_micro & test2_micro & train_micro & test1_micro & test2_micro \\
key & p1 & p1 & p1 & p2 & p2 & p2 & p3 & p3 & p3 \\
activation_function &  &  &  &  &  &  &  &  &  \\
\midrule
ReLU & 95.6\% & 94.9\% & 72.0\% & 69.1\% & 70.3\% & 65.1\% & 69.5\% & 68.8\% & 74.4\% \\
ReLU6 & 99.1\% & 99.0\% & 76.7\% & 69.5\% & 70.4\% & 62.9\% & 69.1\% & 68.7\% & 72.2\% \\
trReLU & 95.4\% & 96.1\% & 78.6\% & 76.3\% & 77.7\% & 46.5\% & 72.5\% & 71.7\% & 69.7\% \\
LeakyReLU & 94.6\% & 93.4\% & 69.3\% & 69.3\% & 70.6\% & 62.8\% & 69.3\% & 68.8\% & 74.3\% \\
GELU & 99.3\% & 98.7\% & 83.7\% & 76.0\% & 77.6\% & 50.8\% & 73.1\% & 72.1\% & 75.9\% \\
SiLU & 99.7\% & 99.9\% & 91.0\% & 73.6\% & 75.3\% & 49.4\% & 71.5\% & 70.6\% & 72.2\% \\
Sigmoid & 97.8\% & 97.6\% & 82.9\% & 71.6\% & 73.2\% & 54.3\% & 70.6\% & 70.1\% & 61.9\% \\
Normalized & 96.3\% & 95.4\% & 74.2\% & 70.8\% & 72.0\% & 52.5\% & 70.6\% & 70.2\% 

## [dPTQ] Generalization Ratio and Gap


In [295]:
df_generalization_ratio = df_qua_selected.drop(columns=['test1_loss','test2_loss','train_macro','test1_macro','test2_macro'])

df_generalization_ratio['GR_Test1']=df_generalization_ratio['test1_micro'].mul(100)/df_generalization_ratio['train_micro'].mul(100)
# df_generalization_ratio['GR_Test1']=df_generalization_ratio['GR_Test1'].round(2)
# df_generalization_ratio['GR_error_Test1']=(1-df_generalization_ratio['test1_micro'])/(1-df_generalization_ratio['train_micro'])
df_generalization_ratio['GGap_Test1']=df_generalization_ratio['train_micro'].mul(100)-df_generalization_ratio['test1_micro'].mul(100)

df_generalization_ratio['GR_Test2']=df_generalization_ratio['test2_micro'].mul(100)/df_generalization_ratio['train_micro'].mul(100)
# df_generalization_ratio['GR_Test2']=df_generalization_ratio['GR_Test2'].round(2)
# df_generalization_ratio['GR_error_Test2']=(1-df_generalization_ratio['test2_micro'])/(1-df_generalization_ratio['train_micro'])
df_generalization_ratio['GGap_Test2']=df_generalization_ratio['train_micro'].mul(100)-df_generalization_ratio['test2_micro'].mul(100)


df_generalization_ratio_all_l_qua=df_generalization_ratio.drop(columns=['train_micro','test1_micro','test2_micro'])

## [dPTQ]. LaTex tables "Generalization performance of the #-layer ACR-GNN with different activation functions (A/F), reported as both Generalization Ratios (Test/Train) and Generalization Gaps (Train – Test accuracy)."

In [296]:
activation_order = ['ReLU', 'ReLU6', 'trReLU', 'LeakyReLU', 'GELU', 'SiLU', 'Sigmoid','Normalized', 'Softplus', 'ELU']
cols = ["GR_Test1", "GGap_Test1", "GR_Test2","GGap_Test2"]
for i in range(1,11):
    df_generalization_ratio_qua=df_generalization_ratio_all_l_qua[(df_generalization_ratio_all_l_qua['layer']==i) ].copy()
    # Make 'activation_function' a categorical column with this order
    df_generalization_ratio_qua["activation_function"] = pd.Categorical(
        df_generalization_ratio_qua["activation_function"],
        categories=activation_order,
        ordered=True
    )
    
    # Sort according to the defined order
    df_generalization_ratio_qua = df_generalization_ratio_qua.sort_values("activation_function")
    
    
    
    # Pivot so that activation_function is index and keys become columns
    table_df_qua = df_generalization_ratio_qua.pivot_table(
        index="activation_function",
        columns="key",
        values=cols,
        observed=False
    )
    #print(table_df)
    # Reorder columns so p1, p2, p3 are grouped together
    table_df_qua = table_df_qua[[("GR_Test1", "p1"), ("GGap_Test1", "p1"),("GR_Test2", "p1"), ("GGap_Test2", "p1"),
                         ("GR_Test1", "p2"), ("GGap_Test1", "p2"),("GR_Test2", "p2"), ("GGap_Test2", "p2"),
                         ("GR_Test1", "p3"), ("GGap_Test1", "p3"),("GR_Test2", "p3"), ("GGap_Test2", "p3")]].round(4)
    
    # Convert to LaTeX
    latex_str = table_df_qua.to_latex(
        index=True,
        header=True,
        multicolumn=True,
        multirow=False,
        escape=False,
        column_format="c ccc ccc ccc ccc",
        caption=f"Generalization performance of the {i}-layer ACR-GNN with different activation functions (A/F), reported as both Generalization Ratios (Test/Train) and Generalization Gaps (Train – Test accuracy) across datasets $p_1$, $p_2$, and $p_3$.",
        label=f"tab:results:nonqua:ACR-GNN:{i}layer",
        float_format="%.3f"  # formats numbers like 96.9%
    )
    print("----", i)
    print(latex_str)

---- 1
\begin{table}
\caption{Generalization performance of the 1-layer ACR-GNN with different activation functions (A/F), reported as both Generalization Ratios (Test/Train) and Generalization Gaps (Train – Test accuracy) across datasets $p_1$, $p_2$, and $p_3$.}
\label{tab:results:nonqua:ACR-GNN:1layer}
\begin{tabular}{c ccc ccc ccc ccc}
\toprule
 & GR_Test1 & GGap_Test1 & GR_Test2 & GGap_Test2 & GR_Test1 & GGap_Test1 & GR_Test2 & GGap_Test2 & GR_Test1 & GGap_Test1 & GR_Test2 & GGap_Test2 \\
key & p1 & p1 & p1 & p1 & p2 & p2 & p2 & p2 & p3 & p3 & p3 & p3 \\
activation_function &  &  &  &  &  &  &  &  &  &  &  &  \\
\midrule
ReLU & 0.993 & 0.689 & 0.753 & 23.599 & 1.017 & -1.175 & 0.942 & 4.007 & 0.991 & 0.639 & 1.071 & -4.943 \\
ReLU6 & 0.999 & 0.104 & 0.774 & 22.388 & 1.013 & -0.900 & 0.904 & 6.656 & 0.995 & 0.325 & 1.046 & -3.163 \\
trReLU & 1.008 & -0.752 & 0.824 & 16.752 & 1.019 & -1.431 & 0.610 & 29.779 & 0.989 & 0.811 & 0.961 & 2.797 \\
LeakyReLU & 0.987 & 1.256 & 0.732 & 25.33

---- 5
\begin{table}
\caption{Generalization performance of the 5-layer ACR-GNN with different activation functions (A/F), reported as both Generalization Ratios (Test/Train) and Generalization Gaps (Train – Test accuracy) across datasets $p_1$, $p_2$, and $p_3$.}
\label{tab:results:nonqua:ACR-GNN:5layer}
\begin{tabular}{c ccc ccc ccc ccc}
\toprule
 & GR_Test1 & GGap_Test1 & GR_Test2 & GGap_Test2 & GR_Test1 & GGap_Test1 & GR_Test2 & GGap_Test2 & GR_Test1 & GGap_Test1 & GR_Test2 & GGap_Test2 \\
key & p1 & p1 & p1 & p1 & p2 & p2 & p2 & p2 & p3 & p3 & p3 & p3 \\
activation_function &  &  &  &  &  &  &  &  &  &  &  &  \\
\midrule
ReLU & 0.997 & 0.215 & 0.975 & 1.717 & 1.010 & -0.825 & 0.977 & 1.910 & 0.981 & 1.463 & 1.001 & -0.071 \\
ReLU6 & 0.990 & 0.630 & 0.962 & 2.438 & 1.014 & -0.985 & 0.939 & 4.145 & 0.978 & 1.479 & 1.053 & -3.481 \\
trReLU & 0.989 & 0.580 & 0.964 & 1.902 & 1.038 & -2.441 & 0.800 & 12.736 & 0.984 & 0.957 & 0.668 & 19.483 \\
LeakyReLU & 0.998 & 0.116 & 0.973 & 1.846 & 

In [297]:
cols_to_compare = ["train_micro", "test1_micro", "test2_micro"]

# Align DataFrames by index (to make sure subtraction is valid)
df1 = df_selected.set_index(["activation_function", "key", "layer"])
df2 = df_qua_heatmap.set_index(["activation_function", "key", "layer"])

# Calculate differences only for selected columns
df_diff = df1[cols_to_compare] - df2[cols_to_compare]
df_diff= df_diff[cols_to_compare].mul(100)
# Reset index if needed
df_diff = df_diff.reset_index()

df_diff.head()

,activation_function,key,layer,train_micro,test1_micro,test2_micro
0,ReLU,p1,1,0.247016,0.468653,1.311381
1,ReLU,p1,2,0.000000,0.000000,-0.057643
2,ReLU,p1,3,0.107515,0.291803,0.281010
3,ReLU,p1,4,0.020881,0.132638,0.371078
4,ReLU,p1,5,-0.020881,-0.066318,-0.216162


In [298]:
df_diff[df_diff.layer==1]

,activation_function,key,layer,train_micro,test1_micro,test2_micro
0,ReLU,p1,1,0.247016,0.468653,1.311381
10,ReLU,p2,1,0.107001,0.013370,-1.902223
20,ReLU,p3,1,-0.134350,-0.217517,-0.050523
30,ReLU6,p1,1,-0.138614,-0.561499,-2.125590
40,ReLU6,p2,1,0.023531,0.138153,-0.695320
50,ReLU6,p3,1,-0.046711,-0.199760,0.129917
60,trReLU,p1,1,0.638423,-0.623397,-0.021616
70,trReLU,p2,1,0.197575,-0.022283,4.434918
80,trReLU,p3,1,-0.031140,0.275226,-0.523276
90,LeakyReLU,p1,1,0.279449,0.813512,0.277407


In [299]:
activation_order = ['ReLU', 'ReLU6', 'trReLU', 'LeakyReLU', 'GELU', 'SiLU', 'Sigmoid','Normalized', 'Softplus', 'ELU']
cols = ["train_micro", "test1_micro", "test2_micro"]
for i in range(2,11):
    df_diff_=df_diff[(df_diff['layer']==i) ].copy()
    # Make 'activation_function' a categorical column with this order
    df_diff_["activation_function"] = pd.Categorical(
        df_diff_["activation_function"],
        categories=activation_order,
        ordered=True
    )
    
    # Sort according to the defined order
    df_diff_ = df_diff_.sort_values("activation_function")
    
    
    
    # Pivot so that activation_function is index and keys become columns
    table_df = df_diff_.pivot_table(
        index="activation_function",
        columns="key",
        values=cols,
        observed=False
    )
    #print(table_df)
    
    # Convert to LaTeX
    delta="$\Delta_{acc}$, \%"
    latex_str = table_df.to_latex(
        index=True,
        header=True,
        multicolumn=True,
        multirow=False,
        escape=False,
        column_format="c ccc ccc ccc ccc",
        caption=f"Accuracy differences ({delta}) of the {i}-layer ACR-GNN with different activation functions (A/F) across datasets $p_1$, $p_2$, and $p_3$.",
        label=f"tab:results:nonqua:ACR-GNN:{i}layer",
        float_format="%.3f"  # formats numbers like 96.9%
    )
    print("----", i)
    print(latex_str)

---- 2
\begin{table}
\caption{Accuracy differences ($\Delta_{acc}$, \%) of the 2-layer ACR-GNN with different activation functions (A/F) across datasets $p_1$, $p_2$, and $p_3$.}
\label{tab:results:nonqua:ACR-GNN:2layer}
\begin{tabular}{c ccc ccc ccc ccc}
\toprule
 & \multicolumn{3}{r}{test1_micro} & \multicolumn{3}{r}{test2_micro} & \multicolumn{3}{r}{train_micro} \\
key & p1 & p2 & p3 & p1 & p2 & p3 & p1 & p2 & p3 \\
activation_function &  &  &  &  &  &  &  &  &  \\
\midrule
ReLU & 0.000 & 0.312 & 0.107 & -0.058 & 0.310 & -0.036 & 0.000 & 0.193 & 0.114 \\
ReLU6 & 0.000 & 0.236 & 0.302 & 0.703 & -0.133 & 0.022 & 0.000 & 0.154 & 0.281 \\
trReLU & -0.013 & 1.769 & 0.435 & -0.047 & 2.572 & 2.494 & 0.093 & 0.900 & 0.738 \\
LeakyReLU & 0.000 & 0.357 & 0.280 & 0.155 & 1.452 & -0.011 & 0.001 & 0.101 & 0.147 \\
GELU & 0.000 & 1.368 & 0.226 & 1.059 & 0.782 & 0.022 & 0.006 & 0.631 & 0.189 \\
SiLU & 0.000 & 0.593 & 0.138 & 0.231 & 0.274 & 0.014 & -0.000 & 0.156 & 0.142 \\
Sigmoid & 0.009 & 0.891